In [ ]:
import os
import json
import numpy as np
import tiktoken
from typing import List, Dict, Any, Tuple, Literal
from dataclasses import dataclass
import time
from openai import OpenAI
from enum import Enum


class SearchMode(Enum):
    TEXT = "text"
    EMBEDDING = "embedding"
    COMBINED = "combined"


@dataclass
class CONSTANTS:
    max_search_top_k: int = 10
    graph_database_save_dir: str = "./context_database"
    # Weight for combining scores (embedding_weight)
    # Higher value gives more importance to embedding similarity
    embedding_weight: float = 0.7


class CodexTokenizer:
    def __init__(self):
        self.tokenizer = tiktoken.encoding_for_model("gpt-3.5")

    def tokenize(self, text: str) -> List[int]:
        return self.tokenizer.encode(text, allowed_special={"<|endoftext|>"})

    def decode(self, token_ids: List[int]) -> str:
        return self.tokenizer.decode(token_ids)


class DatabricksEmbedding:
    def __init__(self, databricks_token=None, base_url=None):
        self.token = databricks_token or os.environ.get("DATABRICKS_TOKEN")
        self.base_url = (
            base_url
            or "https://adb-379144824042062.2.azuredatabricks.net/serving-endpoints"
        )

        self.client = OpenAI(api_key=self.token, base_url=self.base_url)

    def get_embedding(
        self, text: str, model: str = "databricks-bge-large-en"
    ) -> List[float]:
        response = self.client.embeddings.create(input=text, model=model)
        return response.data[0].embedding


class SimilarityMetrics:
    @staticmethod
    def text_jaccard_similarity(list1: List[int], list2: List[int]) -> float:
        set1 = set(list1)
        set2 = set(list2)
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return float(intersection) / union if union > 0 else 0.0

    @staticmethod
    def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
        vec1 = np.array(vec1)
        vec2 = np.array(vec2)

        if np.all(vec1 == 0) or np.all(vec2 == 0):
            return 0.0

        cosine_sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
        return float(cosine_sim)


class CodeSearchInference:
    def __init__(
        self,
        database_dir: str = CONSTANTS.graph_database_save_dir,
        databricks_token: str = None,
        databricks_url: str = None,
    ):
        self.database_dir = database_dir
        self.constants = CONSTANTS()
        self.tokenizer = CodexTokenizer()
        self.embedding_model = DatabricksEmbedding(
            databricks_token=databricks_token, base_url=databricks_url
        )

    def load_database(self, repo_name: str) -> List[Dict[str, Any]]:
        """Load the code database for a specific repository"""
        database_path = os.path.join(self.database_dir, f"{repo_name}.jsonl")
        if not os.path.exists(database_path):
            raise FileNotFoundError(f"Database file not found: {database_path}")

        database = []
        with open(database_path, "r", encoding="utf-8") as f:
            for line in f:
                database.append(json.loads(line.strip()))
        return database

    def _get_text_similarity_results(
        self, query: str, database: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Get results using text-based Jaccard similarity"""
        query_tokens = self.tokenizer.tokenize(query)
        results = []

        for case in database:
            sim_score = SimilarityMetrics.text_jaccard_similarity(
                query_tokens, case["key_forward_encoding"]
            )
            if sim_score > 0:
                result = {**case, "text_similarity": sim_score}
                results.append(result)

        return results

    def _get_embedding_similarity_results(
        self, query: str, database: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Get results using embedding-based cosine similarity"""
        query_embedding = self.embedding_model.get_embedding(query)
        results = []

        for case in database:
            sim_score = SimilarityMetrics.cosine_similarity(
                query_embedding, case["key_forward_embedding"]
            )
            if sim_score > 0:
                result = {**case, "embedding_similarity": sim_score}
                results.append(result)

        return results

    def search(
        self, query: str, repo_name: str, mode: SearchMode = SearchMode.COMBINED
    ) -> Tuple[List[Dict[str, Any]], Dict[str, float]]:
        """
        Search for code snippets using specified search mode

        Args:
            query: Natural language query
            repo_name: Name of the repository to search in
            mode: Search mode (text, embedding, or combined)

        Returns:
            Tuple of (list of results, timing information)
        """
        timing = {}
        start_time = time.time()

        try:
            database = self.load_database(repo_name)
        except FileNotFoundError as e:
            print(f"Error: {e}")
            return [], {"total_time": 0.0}

        results = []
        if mode in [SearchMode.TEXT, SearchMode.COMBINED]:
            text_start = time.time()
            text_results = self._get_text_similarity_results(query, database)
            timing["text_search_time"] = time.time() - text_start

            if mode == SearchMode.TEXT:
                results = [
                    {
                        "val": r["val"],
                        "statement": r["statement"],
                        "key_forward_context": r["key_forward_context"],
                        "key_forward_graph": r["key_forward_graph"],
                        "fpath_tuple": r["fpath_tuple"],
                        "similarity_score": r["text_similarity"],
                    }
                    for r in text_results
                ]

        if mode in [SearchMode.EMBEDDING, SearchMode.COMBINED]:
            embedding_start = time.time()
            embedding_results = self._get_embedding_similarity_results(query, database)
            timing["embedding_search_time"] = time.time() - embedding_start

            if mode == SearchMode.EMBEDDING:
                results = [
                    {
                        "val": r["val"],
                        "statement": r["statement"],
                        "key_forward_context": r["key_forward_context"],
                        "key_forward_graph": r["key_forward_graph"],
                        "fpath_tuple": r["fpath_tuple"],
                        "similarity_score": r["embedding_similarity"],
                    }
                    for r in embedding_results
                ]

        if mode == SearchMode.COMBINED:
            # Create a map of results by their content for combining scores
            combined_results = {}

            # Process text results
            for r in text_results:
                key = (r["val"], r["statement"])  # Use content as key
                combined_results[key] = {
                    "val": r["val"],
                    "statement": r["statement"],
                    "key_forward_context": r["key_forward_context"],
                    "key_forward_graph": r["key_forward_graph"],
                    "fpath_tuple": r["fpath_tuple"],
                    "text_similarity": r["text_similarity"],
                    "embedding_similarity": 0.0,
                }

            # Process embedding results
            for r in embedding_results:
                key = (r["val"], r["statement"])
                if key in combined_results:
                    combined_results[key]["embedding_similarity"] = r[
                        "embedding_similarity"
                    ]
                else:
                    combined_results[key] = {
                        "val": r["val"],
                        "statement": r["statement"],
                        "key_forward_context": r["key_forward_context"],
                        "key_forward_graph": r["key_forward_graph"],
                        "fpath_tuple": r["fpath_tuple"],
                        "text_similarity": 0.0,
                        "embedding_similarity": r["embedding_similarity"],
                    }

            # Combine scores using weighted average
            w = self.constants.embedding_weight
            results = [
                {
                    **r,
                    "similarity_score": (
                        w * r["embedding_similarity"] + (1 - w) * r["text_similarity"]
                    ),
                    "text_similarity": r["text_similarity"],
                    "embedding_similarity": r["embedding_similarity"],
                }
                for r in combined_results.values()
            ]

        # Sort by similarity score and get top-k results
        results.sort(key=lambda x: x["similarity_score"], reverse=True)
        results = results[: self.constants.max_search_top_k]

        timing["total_time"] = time.time() - start_time
        return results, timing


def format_results(
    results: List[Dict[str, Any]], timing: Dict[str, float], mode: SearchMode
) -> str:
    """Format search results for display"""
    output = f"Search completed in {timing['total_time']:.4f} seconds\n"
    if "text_search_time" in timing:
        output += f"Text search time: {timing['text_search_time']:.4f} seconds\n"
    if "embedding_search_time" in timing:
        output += (
            f"Embedding search time: {timing['embedding_search_time']:.4f} seconds\n"
        )
    output += f"Found {len(results)} relevant code snippets\n\n"

    for i, result in enumerate(results, 1):
        output += f"Result {i} (Overall Similarity: {result['similarity_score']:.4f})\n"
        if mode == SearchMode.COMBINED:
            output += f"Text Similarity: {result['text_similarity']:.4f}\n"
            output += f"Embedding Similarity: {result['embedding_similarity']:.4f}\n"
        output += f"File: {'/'.join(result['fpath_tuple'])}\n"
        output += "Code Context:\n"
        output += result["val"] + "\n"
        output += "-" * 80 + "\n"

    return output

In [ ]:
# __init__
databricks_token = ???

searcher = CodeSearchInference(databricks_token=databricks_token)

In [ ]:
query = """ 
Function to PowerDown device, set vih1 to 0
"""
repo_name = "versa"

results, timing = searcher.search(query, repo_name, mode=SearchMode.EMBEDDING)